In [64]:
import torch
import torchvision
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets


import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


Take in fixed sized 224x224 rgb image.

Only preprocessing is subtracting mean rbg value, computed on training set from each pixel.

Passed through a stack of conv layers (3x3). 

One config also uses 1x1 as linear transformation of input channels. 

Conv stride fixed to 1 pixel. Padding is 1 pixel for 3x3.

Spatial pooling carried out using 5 max-ooling layers which follow some conv layers. Max pooling performed over 2x2 window with stride 2.

Stack of conv layers followed by fully connected layers. 
First two has 4096 channels, third perfomrs 1000-way ILSVRC classification and thus contains 1000 channels (one for each class).

Final layer is softmax.

All hidden layers equipped with ReLU non-linearity. 

In [65]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

dataset = datasets.CIFAR10('./data', download=True, train=True, transform=transform)
test_data = datasets.CIFAR10('./data', download=True, train=False, transform=transform)



In [66]:
train_data, val_data = random_split(
    dataset,
    [45000, 5000],
    generator=torch.Generator().manual_seed(42)
)

In [67]:
def get_dataloaders(batch_size=32):
    transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
        )
    ])

    dataset = datasets.CIFAR10('./data', download=True, train=True, transform=transform)
    test_data = datasets.CIFAR10('./data', download=True, train=False, transform=transform)

    train_data, val_data = random_split(
        dataset,
        [45000, 5000],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(
        dataset=train_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        dataset=test_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        dataset=val_data,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    return train_loader, test_loader, val_loader

In [68]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            stride=1,
            padding=1  
        )
        self.block = nn.Sequential(
            nn.Conv2d(
            in_channels=in_channels,
            out_channels=out_channels,
            kernel_size=3,
            padding=1,
            stride=1
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )


    def forward(self, x):
        return self.block(x)

In [69]:
class VGGNet(nn.Module):
    def __init__(self, block, layers, in_channels=3, num_classes=10):
        super().__init__()

        self.in_channels = in_channels
        
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1])
        self.layer3 = self._make_layer(block, 256, layers[2])
        self.layer4 = self._make_layer(block, 512, layers[3])
        self.layer5 = self._make_layer(block, 512, layers[4])

        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        out = self.maxpool(self.layer1(x))
        out = self.maxpool(self.layer2(out))
        out = self.maxpool(self.layer3(out))
        out = self.maxpool(self.layer4(out))
        out = self.layer5(out)
        out = self.adaptive_pool(out)

        out = torch.flatten(out, 1)
        out = self.classifier(out)

        return out


    def _make_layer(self, block, out_channels, num_blocks):

        layers = []
        layers.append(block(self.in_channels, out_channels))
        self.in_channels = out_channels

        for i in range(num_blocks - 1):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers) 

In [70]:
def train_epoch(model, loader, criterion, optimizer, epoch, num_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for batch_idx, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        output = model(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        pred = output.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 500 == 0:
            print(f"Epoch: {epoch}/{num_epochs}, Step: {batch_idx+1}/{len(loader)}")
            print(f"Loss: {running_loss / (batch_idx + 1)}\n")

    accuracy = correct / total
    avg_loss = running_loss / len(loader)

    return avg_loss, accuracy

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        output = model(images)
        loss = criterion(output, labels)

        running_loss += loss.item() * images.size(0)
        pred = output.argmax(dim=1)
        correct += (pred == labels).sum().item()
        total += labels.size(0)

    avg_loss = running_loss / len(loader)
    accuracy = correct / total

    return avg_loss, accuracy


In [ ]:
def train(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs=10):
    model.train()
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    best_val_acc = 0.0

    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = train_epoch(
            model, 
            train_loader,
            criterion,
            optimizer,
            epoch,
            num_epochs
        )

        val_loss, val_acc = evaluate(
            model, 
            test_loader,
            criterion
        )

        scheduler.step(val_loss)

        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f"Train loss: {train_loss} | Train_accuracy: {train_acc * 100}%")
        print(f"Validation loss: {val_loss} | Validation accuracy: {val_acc * 100}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'vggnet.pth')
            print(f"New best model saved: (val acc: {best_val_acc})")

    print(f"Best validation accuracy: {best_val_acc}")
    return train_losses, val_losses, train_accs, val_accs


In [73]:
def main():
    model = VGGNet(ConvBlock, [2, 2, 2, 2, 2]).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=1e-1, momentum=0.9, weight_decay=1e-4)
    train_loader, test_loader, val_loader = get_dataloaders(batch_size=32)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.1, threshold=1e-4, cooldown=0, min_lr=1e-6
    )

    train_loss, val_loss, train_acc, val_acc = train(
        model,
        train_loader,
        test_loader,
        criterion,
        optimizer,
        scheduler,
        num_epochs=30
    ) 

    print("Training complete.")

if __name__ == '__main__':
    main()


Epoch: 1/30, Step: 1/1407
Loss: 72.62022399902344

Epoch: 1/30, Step: 501/1407
Loss: 67.62664600760637

Epoch: 1/30, Step: 1001/1407
Loss: 64.84868543703001

Train loss: 63.08 | Train_accuracy: 0.23
Validation loss: 72.85545263808375 | Validation accuracy: 0.235
New best model saved: (val acc: 0.235)
Epoch: 2/30, Step: 1/1407
Loss: 67.39932250976562

Epoch: 2/30, Step: 501/1407
Loss: 55.41552604934174

Epoch: 2/30, Step: 1001/1407
Loss: 53.72917403946151

Train loss: 52.55 | Train_accuracy: 0.37
Validation loss: 59.96016064238624 | Validation accuracy: 0.3812
New best model saved: (val acc: 0.3812)
Epoch: 3/30, Step: 1/1407
Loss: 56.837406158447266

Epoch: 3/30, Step: 501/1407
Loss: 46.75860106873655

Epoch: 3/30, Step: 1001/1407
Loss: 45.35418359335367

Train loss: 44.49 | Train_accuracy: 0.48
Validation loss: 43.14766593786855 | Validation accuracy: 0.5173
New best model saved: (val acc: 0.5173)
Epoch: 4/30, Step: 1/1407
Loss: 40.39640426635742

Epoch: 4/30, Step: 501/1407
Loss: 40.5